In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import torch
torch.cuda.is_available()

True

# Generate a mock dataset

In [4]:
import random
random.seed(55)
# why do we need this part? Can we make it more efficient or make it in pandas dataframe in one go?
CATALOG = {
    1: ("SmartTerm 20", "Term"),
    2: ("Term100 Protect", "Term"),
    3: ("LegacyGold Whole Life", "WholeLife"),
    4: ("Prestige Whole Life", "WholeLife"),
    5: ("WealthBuilder Endowment", "Endowment"),
    6: ("SavingsPlan Endowment", "Endowment"),
    7: ("FlexiInvest Linked", "ILP"),
    8: ("CI Shield", "CriticalIllness"),
}

ITEM_IDS_BY_CAT = {}
for item_id, (_, cat) in CATALOG.items():
    ITEM_IDS_BY_CAT.setdefault(cat, []).append(item_id)

ITEM_IDS_BY_CAT

{'Term': [1, 2],
 'WholeLife': [3, 4],
 'Endowment': [5, 6],
 'ILP': [7],
 'CriticalIllness': [8]}

In [5]:
COVERAGE_MULTIPLE = {
    "Term": (150, 300),
    "WholeLife": (40, 80),
    "Endowment": (3, 8),
    "ILP": (10, 30),
    "CriticalIllness": (50, 100),
}

In [6]:
BASE_COVERAGE = {
    "Term": 1.5,
    "WholeLife": 1.0,
    "Endowment": 0.4,
    "ILP": 0.6,
    "CriticalIllness": 0.3
}

In [7]:
def age_factor(age:float, factor_a:int=25, factor_b:int=45, offset:float=0.8)->float:
    return 1.0 + max(0.0, (age-factor_a)/factor_b) * offset

In [8]:
def sample_category(age: float)->str:
    weights = {
        "Term": max(0.1, 1.5-age/40),
        "CriticalIllness": max(0.1, 1.2-age/50),
        "WholeLife": min(1.5, age/35),
        "Endowment": min(1.3, age/45),
        "ILP": 0.7
    }
    cats = list(weights.keys())
    probs = list(weights.values())
    return random.choices(cats, weights=probs, k=1)[0]

In [9]:
def sample_item_in_category(cat: str)->int:
    return random.choice(ITEM_IDS_BY_CAT[cat])

In [10]:
def lognormal_noise(sigma: float=0.15)->float:
    return float(torch.exp(torch.randn(1) *sigma))

In [11]:
def generate_customer(customer_id: int, n_events: int | None = None, max_events:int = 8):
    if n_events is None:
        n_events = random.randint(2, max_events)
    wealth = float(torch.exp(torch.randn(1) * 0.5 + 2.0))
    age = float(random.randint(22, 55))
    items, ages, prices, sum_insures = [], [], [], []
    for _ in range(n_events):
        cat = sample_category(age)
        item_id = sample_item_in_category(cat)
        low, high = COVERAGE_MULTIPLE[cat]
        multiple = random.uniform(low, high)
        sum_insure = wealth * BASE_COVERAGE[cat] * 10_000 * lognormal_noise()
        price = sum_insure / (multiple * age_factor(age)) * lognormal_noise()

        items.append(item_id)
        ages.append(age)
        prices.append(price)
        sum_insures.append(sum_insure)

        age += random.randint(1, 5)
    return items, ages, prices, sum_insures

In [12]:
def generate_dataset(n_customers: int = 500):
    return [generate_customer(customer_id=i) for i in range(n_customers)]

In [13]:
items, ages, prices, sum_insures = generate_customer(customer_id=1, n_events=6)

for item_id, age, price, sum_insure in zip(items, ages, prices, sum_insures):
    name, cat = CATALOG[item_id]
    ratio = sum_insure / price
    print(f"age {age:>4.0f} | {name:<24} ({cat:<15}) | "
            f"price {price:>10,.0f} | sum_insure {sum_insure:>12,.0f} | ratio {ratio:>6.1f}x")

age   27 | FlexiInvest Linked       (ILP            ) | price      4,108 | sum_insure       65,821 | ratio   16.0x
age   30 | FlexiInvest Linked       (ILP            ) | price      3,625 | sum_insure       66,469 | ratio   18.3x
age   33 | FlexiInvest Linked       (ILP            ) | price      3,007 | sum_insure       69,557 | ratio   23.1x
age   37 | Prestige Whole Life      (WholeLife      ) | price        932 | sum_insure       86,918 | ratio   93.2x
age   41 | FlexiInvest Linked       (ILP            ) | price      2,191 | sum_insure       57,936 | ratio   26.4x
age   42 | FlexiInvest Linked       (ILP            ) | price      1,424 | sum_insure       53,586 | ratio   37.6x


# Construct it into pytorch dataset and dataloader

In [14]:
import torch
from torch.utils.data import Dataset, DataLoader, random_split

MAX_EVENTS = 8

def pad(seq, max_len, pad_value:int | float =0):
    return seq[:max_len] + ([pad_value] * max(0, max_len-len(seq)))

In [15]:
class TransactionDataset(Dataset):
    """Return items, ages, prices, sum_insures"""
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        items, ages, prices, sum_insures = self.samples[idx]
        return (
            torch.tensor(pad(items, MAX_EVENTS), dtype=torch.long),
            torch.tensor(pad(ages, MAX_EVENTS, 0.0), dtype=torch.float).unsqueeze(-1),
            torch.tensor(pad(prices, MAX_EVENTS, 0.0), dtype=torch.float).unsqueeze(-1),
            torch.tensor(pad(sum_insures, MAX_EVENTS, 0.0), dtype=torch.float).unsqueeze(-1),
        )

In [16]:
samples = generate_dataset(n_customers=500)
dataset = TransactionDataset(samples)

n_total = len(dataset)
n_train = int(n_total * .7)
n_val = int(n_total * .15)
n_test = n_total - n_train - n_val

train_set, val_set, test_set = random_split(
    # this approach can be used with Dataset subclass?
    dataset, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(55)
)

len(train_set), len(val_set), len(test_set)

(350, 75, 75)

In [17]:
BATCH_SIZE = 32
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

In [18]:
class Normalizer:
    def __init__(self):
        self.mean:float | None = None
        self.std:float | None = None

    def fit(self, values: torch.Tensor):
        self.mean = values.mean().item()
        self.std = values.std().item()

    def transform(self, values: torch.Tensor) -> torch.Tensor:
        return (values - self.mean) / self.std

    def inverse_transform(self, values: torch.Tensor) -> torch.Tensor:
        return (values * self.std) + self.mean

In [19]:
def collect_train_values(base_dataset, train_subset, field_idx):
    values = []
    for i in train_subset.indices:
        values.extend(base_dataset.samples[i][field_idx])
    return torch.tensor(values, dtype=torch.float)

In [20]:
age_normalizer = Normalizer()
price_normalizer = Normalizer()
sum_insure_normalizer = Normalizer()

In [21]:
age_normalizer.fit(collect_train_values(dataset, train_set, field_idx=1))
price_normalizer.fit(collect_train_values(dataset, train_set, field_idx=2))
sum_insure_normalizer.fit(collect_train_values(dataset, train_set, field_idx=3))

In [22]:
items, ages, prices, sum_insures = next(iter(train_loader))
print(items.shape, ages.shape, prices.shape, sum_insures.shape)

ages_norm = age_normalizer.transform(ages)
print(f"normalized age batch mean ≈ {ages_norm.mean().item():.2f} (should be near 0, not exactly)")

torch.Size([32, 8]) torch.Size([32, 8, 1]) torch.Size([32, 8, 1]) torch.Size([32, 8, 1])
normalized age batch mean ≈ -1.20 (should be near 0, not exactly)


# Fusion design

In [23]:
import torch
import torch.nn as nn
from typing import Literal

class FusionEmbedding(nn.Module):
    def __init__(self, categorical_vocab_sizes:list[int], num_continuous: int, d_model:int, strategy:Literal['sum', 'concat_all', 'concat_emb_summed'] = "sum"):
        super().__init__()
        assert strategy in ("sum", "concat_all", "concat_emb_summed")
        self.strategy = strategy
        self.num_categorical = len(categorical_vocab_sizes)
        self.num_continuous = num_continuous
        self.categorical_embs = nn.ModuleList([
            nn.Embedding(vocab_size+1, d_model, padding_idx=0)
            for vocab_size in categorical_vocab_sizes
        ])
        self.continuous_projs = nn.ModuleList([
            nn.Linear(1, d_model)
            for _ in range(num_continuous)
        ])
        if strategy == "concat_all":
            concat_dim = d_model * (self.num_categorical + self.num_continuous)
            self.project_down = nn.Linear(concat_dim, d_model)

        elif strategy == "concat_emb_summed":
            concat_dim = d_model*2
            self.project_down = nn.Linear(concat_dim, d_model)

        else:
            self.project_down = None
        
    def forward(self, categorical_features: list[torch.Tensor], continuous_features: list[torch.Tensor]) -> torch.Tensor:
        assert len(categorical_features) == self.num_categorical
        assert len(continuous_features) == self.num_continuous

        cat_vecs = [
            emb(feat) for emb, feat in zip(self.categorical_embs, categorical_features)
        ]

        cont_vecs = [
            emb(feat) for emb, feat in zip(self.continuous_projs, continuous_features)
        ]
        
        if self.strategy == "sum":
            return sum(cat_vecs) + sum(cont_vecs)
        
        if self.strategy == "concat_all":
            fused = torch.cat(cat_vecs + cont_vecs, dim=-1)
            return self.project_down(fused)
        
        fused = torch.cat([sum(cat_vecs), sum(cont_vecs)], dim=-1)
        return self.project_down(fused)

In [24]:
batch, seq_len, d_model = 4, 8, 32
categorical_vocab_sizes = [8, 5]   # e.g. item_id (8 items), sales_channel (5 channels)
num_continuous = 3                  # age, price, sum_insure

item_id    = torch.randint(0, categorical_vocab_sizes[0] + 1, (batch, seq_len))
channel_id = torch.randint(0, categorical_vocab_sizes[1] + 1, (batch, seq_len))
age        = torch.randn(batch, seq_len, 1)
price      = torch.randn(batch, seq_len, 1)
sum_insure = torch.randn(batch, seq_len, 1)

for strategy in ["sum", "concat_all", "concat_emb_summed"]:
    fusion = FusionEmbedding(categorical_vocab_sizes, num_continuous, d_model, strategy=strategy)
    out = fusion([item_id, channel_id], [age, price, sum_insure])
    print(f"{strategy:<20} -> {out.shape}")   # expect (4, 8, 32) for every strategy

sum                  -> torch.Size([4, 8, 32])
concat_all           -> torch.Size([4, 8, 32])
concat_emb_summed    -> torch.Size([4, 8, 32])


In [25]:
test_emb = nn.Embedding(10, 10)
test_lin = nn.Linear(1, 10)
items = torch.tensor([1,2,3], dtype=torch.long)
ages = torch.tensor([25, 30, 45], dtype=torch.float).unsqueeze(-1)

In [26]:
iemb = test_emb(items)
alin = test_lin(ages)

In [27]:
iemb.shape, alin.shape

(torch.Size([3, 10]), torch.Size([3, 10]))

In [28]:
(iemb+alin).shape

torch.Size([3, 10])

In [29]:
torch.cat([iemb, alin], dim=-1).shape

torch.Size([3, 20])

# Define loss

In [30]:
import torch.nn as nn

class UncertaintyWeightedLoss(nn.Module):
    def __init__(self, task_types: list[str]):
        super().__init__()
        self.task_types = task_types
        self.log_vars = nn.Parameter(torch.zeros(len(task_types)))

    def forward(self, losses: list[torch.Tensor]) -> torch.Tensor:
        total = torch.tensor(0.0)
        for i, (loss, task_type) in enumerate(zip(losses, self.task_types)):
            precision = torch.exp(-self.log_vars[i]) 
            if task_type == "classification":
                total += (precision * loss) + self.log_vars[i]
            else:
                total += (precision * loss) + (0.5 * self.log_vars[i])
            
        return total

    def precisions(self):
        return {i: torch.exp(-self.log_vars[i]).item() for i in range(len(self.task_types))}

# Encoder Block

## Setup

In [31]:
import torch
import torch.nn as nn
from package.encoder_block import EncoderBlock

class EncoderCLSBackbone(nn.Module):
    def __init__(self, fusion, d_model: int, n_heads: int, max_len: int, num_layers: int =2):
        super().__init__()
        self.fusion = fusion
        self.cls = nn.Parameter(torch.randn(1, 1, d_model))
        self.layers = nn.ModuleList([
            # max_len must be added 1 because we will implement cls prepending
            EncoderBlock(d_model, n_heads, max_len+1) for _ in range(num_layers)
        ])
    
    def forward(self, categorical_features, continuous_features, pad_mask_source):
        batch = pad_mask_source.shape[0]
        pad_mask = pad_mask_source == 0
        cls_col = torch.zeros(batch, 1, dtype=torch.bool, device=pad_mask_source.device)
        pad_mask = torch.concat([cls_col, pad_mask], dim=1)
        x = self.fusion(categorical_features, continuous_features)
        cls = self.cls.expand(batch, -1, -1)
        x = torch.cat([cls, x], dim=1)

        for layer in self.layers:
            x = layer(x, pad_mask)
        
        # return only CLS vector at the 1st position of each sequence
        return x[:, 0]

In [32]:
batch, seq_len, d_model, n_heads = 4, 8, 32, 4
num_items, num_continuous = 8, 3

fusion = FusionEmbedding([num_items], num_continuous, d_model, strategy="sum")
backbone = EncoderCLSBackbone(fusion, d_model=d_model, n_heads=n_heads, max_len=seq_len, num_layers=2)

item_id    = torch.randint(0, num_items + 1, (batch, seq_len))
age        = torch.randn(batch, seq_len, 1)
price      = torch.randn(batch, seq_len, 1)
sum_insure = torch.randn(batch, seq_len, 1)

summary = backbone([item_id], [age, price, sum_insure], pad_mask_source=item_id)
print(summary.shape)   # expect (4, 32) — one vector per customer, actually collapsed this time

torch.Size([4, 32])


In [33]:
def expand_into_prefixes(samples):
    expanded = []
    for items, ages, prices, sum_insures in samples:
        n = len(items)
        for prefix_len in range(1, n):
            expanded.append((
                items[:prefix_len],
                ages[:prefix_len],
                prices[:prefix_len],
                sum_insures[:prefix_len],
                items[prefix_len],
                ages[prefix_len],
                prices[prefix_len],
                sum_insures[prefix_len]
            ))
    return expanded

class NextSequenceDataset(Dataset):
    def __init__(self, expanded_samples):
        self.samples = expanded_samples

    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        items, ages, prices, sum_insures, next_item, next_age, next_price, next_sum_insure = self.samples[idx]
        return (
            torch.tensor(pad(items, MAX_EVENTS), dtype=torch.long),
            torch.tensor(pad(ages, MAX_EVENTS), dtype=torch.float).unsqueeze(-1),
            torch.tensor(pad(prices, MAX_EVENTS), dtype=torch.float).unsqueeze(-1),
            torch.tensor(pad(sum_insures, MAX_EVENTS), dtype=torch.float).unsqueeze(-1),
            torch.tensor(next_item, dtype=torch.long),
            torch.tensor([next_age], dtype=torch.float),
            torch.tensor([next_price], dtype=torch.float),
            torch.tensor([next_sum_insure], dtype=torch.float),
        )

In [34]:
def collect_raw_samples(base_dataset, subset):
    return [base_dataset.samples[i] for i in subset.indices]

train_expanded = expand_into_prefixes(collect_raw_samples(dataset, train_set))
val_expanded = expand_into_prefixes(collect_raw_samples(dataset, val_set))
test_expanded = expand_into_prefixes(collect_raw_samples(dataset, test_set))

In [35]:
next_item_train_loader = DataLoader(NextSequenceDataset(train_expanded), batch_size=BATCH_SIZE, shuffle=True)
next_item_val_loader = DataLoader(NextSequenceDataset(val_expanded), batch_size=BATCH_SIZE, shuffle=False)
next_item_test_loader = DataLoader(NextSequenceDataset(test_expanded), batch_size=BATCH_SIZE, shuffle=False)

print(f"train rows: {len(train_expanded)}  val rows: {len(val_expanded)}  test rows: {len(test_expanded)}")


train rows: 1421  val rows: 276  test rows: 318


In [36]:
class MultiTaskModel(nn.Module):
    def __init__(self, backbone, d_model: int, num_items: int):
        super().__init__()
        self.backbone = backbone
        # num_items + 1 because we reserve 0 for padding
        # item_head will return as logit
        self.item_head = nn.Linear(d_model, num_items+1)
        self.age_head = nn.Linear(d_model, 1)
        self.price_head = nn.Linear(d_model, 1)
        self.sum_insure_head = nn.Linear(d_model, 1)

    def forward(self, categorical_features, continuous_features, pad_mask_source):
        summary = self.backbone(categorical_features, continuous_features, pad_mask_source)
        # is it possible to factor this out into .predict_age_with_constraint?
        ages_norm = continuous_features[0]
        lengths = (pad_mask_source != 0).sum(dim=1)
        batch_idx = torch.arange(ages_norm.shape[0], device=ages_norm.device)
        current_age = ages_norm[batch_idx, lengths-1, 0]
        delta_age = nn.functional.softplus(self.age_head(summary).squeeze(-1))
        age_pred = (current_age + delta_age).unsqueeze(-1)
        return (
            self.item_head(summary),
            age_pred,
            self.price_head(summary),
            self.sum_insure_head(summary),
        )

In [37]:
fusion   = FusionEmbedding([num_items], num_continuous=3, d_model=32, strategy="sum")
backbone = EncoderCLSBackbone(fusion, d_model=32, n_heads=4, max_len=MAX_EVENTS, num_layers=2)
model    = MultiTaskModel(backbone, d_model=32, num_items=num_items)

items, ages, prices, sum_insures, next_item, next_age, next_price, next_sum_insure = next(iter(next_item_train_loader))

# normalize every continuous value — both the input sequence AND the regression
# labels — using the SAME normalizers already fit on the train split only
ages_norm             = age_normalizer.transform(ages)
prices_norm           = price_normalizer.transform(prices)
sum_insures_norm      = sum_insure_normalizer.transform(sum_insures)
next_age_norm         = age_normalizer.transform(next_age)
next_price_norm       = price_normalizer.transform(next_price)
next_sum_insure_norm  = sum_insure_normalizer.transform(next_sum_insure)

item_logits, age_pred, price_pred, sum_insure_pred = model(
    [items], [ages_norm, prices_norm, sum_insures_norm], pad_mask_source=items
)

loss_item       = nn.functional.cross_entropy(item_logits, next_item)
loss_age        = nn.functional.mse_loss(age_pred, next_age_norm)
loss_price      = nn.functional.mse_loss(price_pred, next_price_norm)
loss_sum_insure = nn.functional.mse_loss(sum_insure_pred, next_sum_insure_norm)

print(loss_item.item(), loss_age.item(), loss_price.item(), loss_sum_insure.item())


2.222076654434204 0.17883872985839844 1.937536597251892 1.6146304607391357


In [38]:
loss_weigher = UncertaintyWeightedLoss(task_types=["classification", "regression", "regression", "regression"])

total_loss = loss_weigher([loss_item, loss_age, loss_price, loss_sum_insure])
print(total_loss.item())

5.953082084655762


In [39]:
items.shape, item_logits.shape

(torch.Size([32, 8]), torch.Size([32, 9]))

In [40]:
items[0], (item_logits[0])

(tensor([7, 4, 7, 0, 0, 0, 0, 0]),
 tensor([-1.0100,  0.2127, -0.0479,  0.1024,  0.0877,  0.1516,  0.9831,  0.1959,
         -0.9017], grad_fn=<SelectBackward0>))

In [41]:
next_item[0], torch.argmax(item_logits[0])

(tensor(3), tensor(6))

In [42]:
ages[0].squeeze(-1), age_pred[0], age_normalizer.inverse_transform(age_pred[0]), next_age[0]

(tensor([53., 55., 59.,  0.,  0.,  0.,  0.,  0.]),
 tensor([1.8416], grad_fn=<SelectBackward0>),
 tensor([66.3483], grad_fn=<AddBackward0>),
 tensor([62.]))

## Test train loop

In [43]:
D_MODEL = 32
N_HEADS = 4
NUM_LAYERS = 2

fusion = FusionEmbedding([num_items], num_continuous=3, d_model=D_MODEL, strategy="sum")
backbone = EncoderCLSBackbone(fusion, d_model=D_MODEL, n_heads=N_HEADS, max_len=MAX_EVENTS, num_layers=NUM_LAYERS)
model = MultiTaskModel(backbone, d_model=D_MODEL, num_items=num_items)
loss_weigher = UncertaintyWeightedLoss(task_types=["classification", "regression", "regression", "regression"])

optimizer = torch.optim.Adam(list(model.parameters())+list(loss_weigher.parameters()), lr=1e-3)
EPOCHS = 50

def normalize_batch(ages, prices, sum_insures, next_age, next_price, next_sum_insure):
    return(
        age_normalizer.transform(ages),
        price_normalizer.transform(prices),
        sum_insure_normalizer.transform(sum_insures),
        age_normalizer.transform(next_age),
        price_normalizer.transform(next_price),
        sum_insure_normalizer.transform(next_sum_insure),
    )

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    for items, ages, prices, sum_insures, next_item, next_age, next_price, next_sum_insure in next_item_train_loader:
        ages_norm, prices_norm, sum_insures_norm, next_age_norm, next_price_norm, next_sum_insure_norm = (
            normalize_batch(ages, prices, sum_insures, next_age, next_price, next_sum_insure)
        )
        item_logits, age_pred, price_pred, sum_insure_pred = model(
            [items], [ages_norm, prices_norm, sum_insures_norm], pad_mask_source=items
        )

        loss_item = nn.functional.cross_entropy(item_logits, next_item)
        loss_age = nn.functional.mse_loss(age_pred, next_age_norm)
        loss_price = nn.functional.mse_loss(price_pred, next_price_norm)
        loss_sum_insure = nn.functional.mse_loss(sum_insure_pred, next_sum_insure_norm)

        loss = loss_weigher([loss_item, loss_age, loss_price, loss_sum_insure])

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if (epoch+1) % 5 == 0:
        weights = torch.exp(-loss_weigher.log_vars).detach()
        print(
            f"epoch {epoch+1:3d} | train loss {total_loss / len(next_item_train_loader):4f} "
            f"| weights item={weights[0]:.2f} age={weights[1]:.2f} price={weights[2]:.2f} sum_insure={weights[3]:.2f}"
        )

        model.eval()
        correct, total = 0, 0
        age_abs_err, price_abs_err, sum_insure_abs_err = 0.0, 0.0, 0.0

        with torch.no_grad():
            for items, ages, prices, sum_insures, next_item, next_age, next_price, next_sum_insure in next_item_val_loader:
                ages_norm, prices_norm, sum_insures_norm, *_ = (
                    normalize_batch(ages, prices, sum_insures, next_age, next_price, next_sum_insure)
                )

                item_logits, age_pred, price_pred, sum_insure_pred = model(
                    [items], [ages_norm, prices_norm, sum_insures_norm], pad_mask_source=items
                )

                correct += (item_logits.argmax(dim=-1) == next_item).sum().item()
                total += next_item.numel()

                age_abs_err += (age_normalizer.inverse_transform(age_pred) - next_age).abs().sum().item()
                price_abs_err += (price_normalizer.inverse_transform(price_pred) - next_price).abs().sum().item()
                sum_insure_abs_err += (sum_insure_normalizer.inverse_transform(sum_insure_pred) - next_sum_insure).abs().sum().item()
        n_val = len(next_item_val_loader.dataset)
        print(
            f"    |- val item_acc {correct/total:.2%} "
            f"| age MAE {age_abs_err/n_val:.2f} | price MAE {price_abs_err/n_val:.2f} | sum_insure MAE {sum_insure_abs_err/n_val:.2f}"
        )
        model.train()
    

epoch   5 | train loss 3.065787 | weights item=0.82 age=1.25 price=0.89 sum_insure=0.95
    |- val item_acc 15.94% | age MAE 1.26 | price MAE 1403.57 | sum_insure MAE 26411.04
epoch  10 | train loss 2.775352 | weights item=0.71 age=1.57 price=0.82 sum_insure=0.93
    |- val item_acc 17.39% | age MAE 1.25 | price MAE 1442.10 | sum_insure MAE 25325.60
epoch  15 | train loss 2.572487 | weights item=0.64 age=1.96 price=0.79 sum_insure=0.93
    |- val item_acc 15.94% | age MAE 1.25 | price MAE 1500.00 | sum_insure MAE 28031.51
epoch  20 | train loss 2.336014 | weights item=0.59 age=2.44 price=0.78 sum_insure=0.95
    |- val item_acc 17.39% | age MAE 1.27 | price MAE 1626.05 | sum_insure MAE 26517.03
epoch  25 | train loss 2.105917 | weights item=0.57 age=3.04 price=0.79 sum_insure=1.01
    |- val item_acc 13.41% | age MAE 1.28 | price MAE 1531.85 | sum_insure MAE 27776.01
epoch  30 | train loss 1.858515 | weights item=0.57 age=3.77 price=0.83 sum_insure=1.09
    |- val item_acc 14.49% | age

In [44]:
def theoretical_item_ceiling(age: float) -> float:
    """Bayes-optimal probability of guessing the exact next item correctly,
    using the generator's own category weights — the best any model can do."""
    weights = {
        "Term":            max(0.1, 1.5 - age / 40),
        "CriticalIllness": max(0.1, 1.2 - age / 50),
        "WholeLife":       min(1.5, age / 35),
        "Endowment":       min(1.3, age / 45),
        "ILP":             0.7,
    }
    total = sum(weights.values())
    best_cat, best_cat_weight = max(weights.items(), key=lambda kv: kv[1])

    cat_prob            = best_cat_weight / total                    # P(best category | age)
    item_prob_given_cat = 1 / len(ITEM_IDS_BY_CAT[best_cat])          # uniform pick within that category
    return cat_prob * item_prob_given_cat


# average over the actual ages seen in validation — each val_expanded row is
# (items, ages, prices, sum_insures, next_item, next_age, next_price, next_sum_insure)
current_ages = [ages[-1] for (items, ages, prices, sum_insures, *_ ) in val_expanded]
ceiling = sum(theoretical_item_ceiling(age) for age in current_ages) / len(current_ages)

print(f"theoretical best possible next-item accuracy: {ceiling:.1%}")

theoretical best possible next-item accuracy: 16.8%


## Refactor training process

In [50]:
def build_model(
        architecture: str, 
        fusion_strategy: Literal["sum", "concat_all", "concat_emb_summed"],
        num_items: int,
        num_continuous: int = 3,
        d_model: int = 32,
        n_heads: int = 4,
        num_layers: int = 2,
        max_len: int = 8
):
    fusion = FusionEmbedding([num_items], num_continuous, d_model, strategy=fusion_strategy)

    if architecture == 'encoder':
        backbone = EncoderCLSBackbone(fusion, d_model=d_model, n_heads=n_heads, max_len=max_len, num_layers=num_layers)
    else:
        raise ValueError(f"unknown architecture: {architecture}")

    model = MultiTaskModel(backbone, d_model, num_items)
    loss_weigher = UncertaintyWeightedLoss(task_types=["classification", "regression", "regression", "regression"])
    return model, loss_weigher

In [51]:
def normalize_batch(ages, prices, sum_insures, next_age, next_price, next_sum_insure):
    return (
        age_normalizer.transform(ages),
        price_normalizer.transform(prices),
        sum_insure_normalizer.transform(sum_insures),
        age_normalizer.transform(next_age),
        price_normalizer.transform(next_price),
        sum_insure_normalizer.transform(next_sum_insure),
    )

In [52]:
def compute_losses(
        model,
        items,
        ages_norm,
        prices_norm,
        sum_insures_norm,
        next_item,
        next_age_norm,
        next_price_norm,
        next_sum_insure_norm,
):
    items_logits, age_pred, price_pred, sum_insure_pred = model(
        [items], [ages_norm, prices_norm, sum_insures_norm], pad_mask_source=items
    )
    return (
        nn.functional.cross_entropy(items_logits, next_item),
        nn.functional.mse_loss(age_pred, next_age_norm),
        nn.functional.mse_loss(price_pred, next_price_norm),
        nn.functional.mse_loss(sum_insure_pred, next_sum_insure_norm),
        (items_logits, age_pred, price_pred, sum_insure_pred)
    )

In [58]:
def task_weights(loss_weigher, task_names=("item", "age", "price", "sum_insure")):
    weights = torch.exp(-loss_weigher.log_vars).detach()
    return {name: w.item() for name, w in zip(task_names, weights)}

In [59]:
def run_experiment(architecture, fusion_strategy, epochs=50, patience=3, check_every=5, lr=1e-3, verbose=True):
    tag = f"[{architecture}/{fusion_strategy}]"
    model, loss_weigher = build_model(architecture, fusion_strategy, num_items=num_items)
    optimizer = torch.optim.Adam(list(model.parameters()) + list(loss_weigher.parameters()), lr=lr)

    best_val_loss, best_state_dict, best_metrics = float("inf"), None, None
    check_without_improvement = 0
    history = []

    for epoch in range(epochs):
        model.train()
        for items, ages, prices, sum_insures, next_item, next_age, next_price, next_sum_insure in next_item_train_loader:
            ages_norm, prices_norm, sum_insures_norm, next_age_norm, next_price_norm, next_sum_insure_norm = (
                normalize_batch(ages, prices, sum_insures, next_age, next_price, next_sum_insure)
            )
            loss_item, loss_age, loss_price, loss_sum_insure, _ = compute_losses(
                model, items, ages_norm, prices_norm, sum_insures_norm,
                next_item, next_age_norm, next_price_norm, next_sum_insure_norm
            )
            loss = loss_weigher([loss_item, loss_age, loss_price, loss_sum_insure])

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        
        if (epoch+1) % check_every != 0:
            continue

        model.eval()
        correct, total = 0, 0
        age_err, price_err, sum_insure_err, val_loss_total = 0.0, 0.0, 0.0, 0.0

        with torch.no_grad():
            for items, ages, prices, sum_insures, next_item, next_age, next_price, next_sum_insure in next_item_val_loader:
                ages_norm, prices_norm, sum_insures_norm, next_age_norm, next_price_norm, next_sum_insure_norm = (
                    normalize_batch(ages, prices, sum_insures, next_age, next_price, next_sum_insure)
                )
                loss_item, loss_age, loss_price, loss_sum_insure, (item_logits, age_pred, price_pred, sum_insure_pred) = (
                    compute_losses(
                        model, items, ages_norm, prices_norm, sum_insures_norm,
                        next_item, next_age_norm, next_price_norm, next_sum_insure_norm
                    )
                )
                val_loss_total += loss_weigher([loss_item, loss_age, loss_price, loss_sum_insure]).item()

                correct += ( item_logits.argmax(dim=-1) == next_item ).sum().item()
                total += next_item.numel()
                age_err += (age_normalizer.inverse_transform(age_pred) - next_age).abs().sum().item()
                price_err += (price_normalizer.inverse_transform(price_pred) - next_price).abs().sum().item()
                sum_insure_err += (sum_insure_normalizer.inverse_transform(sum_insure_pred) - next_sum_insure).abs().sum().item()

        n_val = len(next_item_val_loader.dataset)
        current_val_loss = val_loss_total / len(next_item_val_loader)
        metrics = {
            "epoch": epoch+1,
            "val_loss": current_val_loss,
            "item_acc": correct / total,
            "age_mae": age_err / n_val,
            "price_mae": price_err / n_val,
            "sum_insure_mae": sum_insure_err / n_val,
            "weights": task_weights(loss_weigher),
        }
        history.append(metrics)
        if verbose:
            w = metrics["weights"]
            print(
                f"{tag} epoch {epoch+1:3d} | val_loss {current_val_loss:.4f} | item_acc {metrics['item_acc']:.2%} "
                f"| age_mae {metrics['age_mae']:.2f} | price_mae {metrics['price_mae']:.2f} | sum_insure_mae {metrics['sum_insure_mae']:.2f}"
            )
            print(f"{tag}   weights: item={w['item']:.2f} age={w['age']:.2f} price={w['price']:.2f} sum_insure={w['sum_insure']:.2f}")

        if current_val_loss < best_val_loss:
            best_val_loss = current_val_loss
            best_state_dict = {k: v.clone() for k, v in model.state_dict().items()}
            best_metrics = metrics
            checks_without_improvement = 0
        
        else:
            checks_without_improvement += 1
            if checks_without_improvement >= patience:
                if verbose:
                    print(
                        f"{tag} no improvement for {patience} checks - stopping early at epoch {epoch+1}"
                    )
                break
    
    model.load_state_dict(best_state_dict)
    return {
        "architecture": architecture,
        "fusion": fusion_strategy,
        "model": model,
        "best_metrics": best_metrics,
        "history": history
    }

        

In [65]:
result = run_experiment(architecture="encoder", fusion_strategy="sum")
print(result["best_metrics"])

[encoder/sum] epoch   5 | val_loss 3.0245 | item_acc 17.75% | age_mae 1.29 | price_mae 1428.05 | sum_insure_mae 26305.29
[encoder/sum]   weights: item=0.82 age=1.25 price=0.89 sum_insure=0.95
[encoder/sum] epoch  10 | val_loss 2.7634 | item_acc 15.94% | age_mae 1.26 | price_mae 1548.24 | sum_insure_mae 25955.52
[encoder/sum]   weights: item=0.71 age=1.57 price=0.83 sum_insure=0.93
[encoder/sum] epoch  15 | val_loss 2.6685 | item_acc 17.75% | age_mae 1.27 | price_mae 1519.46 | sum_insure_mae 26887.11
[encoder/sum]   weights: item=0.63 age=1.96 price=0.79 sum_insure=0.93
[encoder/sum] epoch  20 | val_loss 2.6297 | item_acc 14.49% | age_mae 1.27 | price_mae 1548.68 | sum_insure_mae 27363.39
[encoder/sum]   weights: item=0.59 age=2.45 price=0.78 sum_insure=0.94
[encoder/sum] epoch  25 | val_loss 2.5446 | item_acc 18.12% | age_mae 1.28 | price_mae 1609.05 | sum_insure_mae 25798.32
[encoder/sum]   weights: item=0.57 age=3.04 price=0.80 sum_insure=0.99
[encoder/sum] epoch  30 | val_loss 2.611

In [66]:
result = run_experiment(architecture="encoder", fusion_strategy="concat_all")
print(result["best_metrics"])

[encoder/concat_all] epoch   5 | val_loss 2.9606 | item_acc 18.12% | age_mae 1.25 | price_mae 1441.40 | sum_insure_mae 27035.45
[encoder/concat_all]   weights: item=0.82 age=1.25 price=0.89 sum_insure=0.95
[encoder/concat_all] epoch  10 | val_loss 2.7577 | item_acc 16.30% | age_mae 1.26 | price_mae 1404.15 | sum_insure_mae 25518.03
[encoder/concat_all]   weights: item=0.71 age=1.57 price=0.82 sum_insure=0.93
[encoder/concat_all] epoch  15 | val_loss 2.6176 | item_acc 16.67% | age_mae 1.27 | price_mae 1462.54 | sum_insure_mae 26800.08
[encoder/concat_all]   weights: item=0.63 age=1.96 price=0.78 sum_insure=0.93
[encoder/concat_all] epoch  20 | val_loss 2.5769 | item_acc 18.12% | age_mae 1.28 | price_mae 1534.48 | sum_insure_mae 26964.43
[encoder/concat_all]   weights: item=0.59 age=2.44 price=0.76 sum_insure=0.95
[encoder/concat_all] epoch  25 | val_loss 2.6395 | item_acc 17.03% | age_mae 1.26 | price_mae 1554.65 | sum_insure_mae 29775.07
[encoder/concat_all]   weights: item=0.57 age=3.

In [67]:
result = run_experiment(architecture="encoder", fusion_strategy="concat_emb_summed")
print(result["best_metrics"])

[encoder/concat_emb_summed] epoch   5 | val_loss 2.9598 | item_acc 18.48% | age_mae 1.25 | price_mae 1428.26 | sum_insure_mae 26392.86
[encoder/concat_emb_summed]   weights: item=0.82 age=1.25 price=0.89 sum_insure=0.94
[encoder/concat_emb_summed] epoch  10 | val_loss 2.7668 | item_acc 17.39% | age_mae 1.26 | price_mae 1473.62 | sum_insure_mae 26877.68
[encoder/concat_emb_summed]   weights: item=0.71 age=1.57 price=0.82 sum_insure=0.92
[encoder/concat_emb_summed] epoch  15 | val_loss 2.6580 | item_acc 17.03% | age_mae 1.25 | price_mae 1494.58 | sum_insure_mae 27664.70
[encoder/concat_emb_summed]   weights: item=0.63 age=1.96 price=0.77 sum_insure=0.92
[encoder/concat_emb_summed] epoch  20 | val_loss 2.5806 | item_acc 17.39% | age_mae 1.27 | price_mae 1457.14 | sum_insure_mae 27113.52
[encoder/concat_emb_summed]   weights: item=0.59 age=2.44 price=0.75 sum_insure=0.94
[encoder/concat_emb_summed] epoch  25 | val_loss 2.5891 | item_acc 15.58% | age_mae 1.28 | price_mae 1496.07 | sum_insur

# Decoder Block

In [68]:
import torch
import torch.nn as nn
from package.gpt_decoder_block import GPTDecoderBlock

class DecoderBackbone(nn.Module):
    def __init__(self, fusion, d_model: int, n_heads: int, max_len: int, num_layers: int = 2):
        super().__init__()
        self.fusion = fusion
        self.layers = nn.ModuleList([
            GPTDecoderBlock(d_model, n_heads, max_len) for _ in range(num_layers)
        ])
    
    def forward(self, categorical_features, continuous_features, pad_mask_source):
        pad_mask = pad_mask_source == 0
        x = self.fusion(categorical_features, continuous_features)

        for layer in self.layers:
            x = layer(x, pad_mask)

        # return shape: (batch, seq_len, d_model)
        return x

In [69]:
class MultiTaskDecoderModel(nn.Module):
    def __init__(self, backbone, d_model: int, num_items: int):
        super().__init__()
        self.backbone = backbone
        self.item_head = nn.Linear(d_model, num_items + 1)
        self.age_head = nn.Linear(d_model, 1)
        self.price_head = nn.Linear(d_model, 1)
        self.sum_insure_head = nn.Linear(d_model, 1)
    
    def forward(self, categorical_features, continuous_features, pad_mask_source):
        x = self.backbone(categorical_features, continuous_features, pad_mask_source)
        ages_norm = continuous_features[0]
        delta_age = nn.functional.softplus(self.age_head(x))
        age_pred = ages_norm + delta_age

        return (
            self.item_head(x),
            age_pred,
            self.price_head(x),
            self.sum_insure_head(x)
        )

In [70]:
batch, seq_len, d_model, n_heads = 4, 8, 32, 4

fusion   = FusionEmbedding([num_items], num_continuous=3, d_model=d_model, strategy="sum")
backbone = DecoderBackbone(fusion, d_model=d_model, n_heads=n_heads, max_len=seq_len, num_layers=2)
model    = MultiTaskDecoderModel(backbone, d_model=d_model, num_items=num_items)

item_id    = torch.randint(0, num_items + 1, (batch, seq_len))
age        = torch.randn(batch, seq_len, 1)
price      = torch.randn(batch, seq_len, 1)
sum_insure = torch.randn(batch, seq_len, 1)

item_logits, age_pred, price_pred, sum_insure_pred = model(
    [item_id], [age, price, sum_insure], pad_mask_source=item_id
)

print(item_logits.shape)      # expect (4, 8, num_items+1)
print(age_pred.shape)         # expect (4, 8, 1)
print(price_pred.shape)       # expect (4, 8, 1)
print(sum_insure_pred.shape)  # expect (4, 8, 1)

torch.Size([4, 8, 9])
torch.Size([4, 8, 1])
torch.Size([4, 8, 1])
torch.Size([4, 8, 1])


In [71]:
def masked_mse(pred, target, mask):
    mask = mask.unsqueeze(-1)
    squared_error = (pred-target) ** 2 * mask
    return squared_error.sum() / mask.sum().clamp(min=1)

In [72]:
items, ages, prices, sum_insures = next(iter(train_loader))

ages_norm = age_normalizer.transform(ages)
prices_norm = price_normalizer.transform(prices)
sum_insures_norm = sum_insure_normalizer.transform(sum_insures)

item_logits, age_pred, price_pred, sum_insure_pred = model(
    [items], [ages_norm, prices_norm, sum_insures_norm], pad_mask_source=items
)

pred_item = item_logits[:, :-1]
pred_age = age_pred[:, :-1]
pred_price = price_pred[:, :-1]
pred_sum_insure = sum_insure_pred[:, :-1]

tgt_item = items[:, 1:]
tgt_age_norm = ages_norm[:, 1:]
tgt_price_norm = prices_norm[:, 1:]
tgt_sum_insure_norm = sum_insures_norm[:, 1:]

loss_item = nn.functional.cross_entropy(pred_item.transpose(1, 2), tgt_item, ignore_index=0)

valid_mask = tgt_item != 0

loss_age = masked_mse(pred_age, tgt_age_norm, valid_mask)
loss_price = masked_mse(pred_price, tgt_price_norm, valid_mask)
loss_sum_insure = masked_mse(pred_sum_insure, tgt_sum_insure_norm, valid_mask)

print(loss_item.item(), loss_age.item(), loss_price.item(), loss_sum_insure.item())

2.2689077854156494 0.3175118863582611 1.2793314456939697 2.0182831287384033


In [92]:
def encoder_batch_step(model, batch):
    items, ages, prices, sum_insures, next_item, next_age, next_price, next_sum_insure = batch
    ages_norm, prices_norm, sum_insures_norm, next_age_norm, next_price_norm, next_sum_insure_norm = (
        normalize_batch(ages, prices, sum_insures, next_age, next_price, next_sum_insure)
    )
    item_logits, age_pred, price_pred, sum_insure_pred = model(
        [items], [ages_norm, prices_norm, sum_insures_norm], pad_mask_source=items
    )

    losses = (
        nn.functional.cross_entropy(item_logits, next_item),
        nn.functional.mse_loss(age_pred, next_age_norm),
        nn.functional.mse_loss(price_pred, next_price_norm),
        nn.functional.mse_loss(sum_insure_pred, next_sum_insure_norm),
    )
    stats = {
        "correct": (item_logits.argmax(dim=-1)==next_item).sum().item(),
        "total": next_item.numel(),
        "age_err": (age_normalizer.inverse_transform(age_pred)-next_age).abs().sum().item(),
        "price_err": (price_normalizer.inverse_transform(price_pred)-next_price).abs().sum().item(),
        "sum_insure_err": (sum_insure_normalizer.inverse_transform(sum_insure_pred)-next_sum_insure).abs().sum().item(),
    }
    return losses, stats

In [93]:
def decoder_batch_step(model, batch):
    items, ages, prices, sum_insures = batch
    ages_norm = age_normalizer.transform(ages)
    prices_norm = price_normalizer.transform(prices)
    sum_insures_norm = sum_insure_normalizer.transform(sum_insures)

    item_logits, age_pred, price_pred, sum_insure_pred = model(
        [items], [ages_norm, prices_norm, sum_insures_norm], pad_mask_source=items
    )

    pred_item, pred_age, pred_price, pred_sum_insure = (
        item_logits[:, :-1], age_pred[:, :-1], price_pred[:, :-1], sum_insure_pred[:, :-1]
    )
    tgt_item = items[:, 1:]
    tgt_age_norm, tgt_price_norm, tgt_sum_insure_norm = ages_norm[:, 1:], prices_norm[:, 1:], sum_insures_norm[:, 1:]
    valid_mask = tgt_item != 0

    losses = (
        nn.functional.cross_entropy(pred_item.transpose(1, 2), tgt_item, ignore_index=0),
        masked_mse(pred_age, tgt_age_norm, valid_mask),
        masked_mse(pred_price, tgt_price_norm, valid_mask),
        masked_mse(pred_sum_insure, tgt_sum_insure_norm, valid_mask),
    )

    tgt_age, tgt_price, tgt_sum_insure = ages[:, 1:], prices[:, 1:], sum_insures[:, 1:]
    mask3 = valid_mask.unsqueeze(-1)
    stats = {
        "correct": ((pred_item.argmax(dim=-1)==tgt_item) & valid_mask).sum().item(),
        "total": valid_mask.sum().item(),
        "age_err": (((age_normalizer.inverse_transform(pred_age)-tgt_age).abs())*mask3).sum().item(),
        "price_err": (((price_normalizer.inverse_transform(pred_price)-tgt_price).abs())*mask3).sum().item(),
        "sum_insure_err": (((sum_insure_normalizer.inverse_transform(pred_sum_insure)-tgt_sum_insure).abs())*mask3).sum().item(),
    }
    return losses, stats

In [94]:
def build_model(
        architecture,
        fusion_strategy,
        num_items,
        num_continuous=3,
        d_model=32,
        n_heads=4,
        num_layers=2,
        max_len=8,
):
    fusion = FusionEmbedding([num_items], num_continuous, d_model, strategy=fusion_strategy)
    if architecture == "encoder":
        backbone = EncoderCLSBackbone(fusion, d_model=d_model, n_heads=n_heads, max_len=max_len, num_layers=num_layers)
        model = MultiTaskModel(backbone, d_model=d_model, num_items=num_items)
    elif architecture == "decoder":
        backbone = DecoderBackbone(fusion, d_model=d_model, n_heads=n_heads, max_len=max_len, num_layers=num_layers)
        model = MultiTaskDecoderModel(backbone, d_model=d_model, num_items=num_items)
    else:
        raise ValueError(f"Unknown architecture: {architecture}")
    
    loss_weigher = UncertaintyWeightedLoss(task_types=["classification", "regression", "regression", "regression"])
    return model, loss_weigher

In [95]:
# this is considered to be a great way to setup parameters
BATCH_STEP = {"encoder": encoder_batch_step, "decoder": decoder_batch_step}
LOADERS = {
    "encoder": (next_item_train_loader, next_item_val_loader),
    "decoder": (train_loader, val_loader)
}

def run_experiment(
        architecture, 
        fusion_strategy,
        epochs=50,
        patience=3,
        check_every=5,
        lr=1e-3,
        verbose=True,
):
    tag = f"[{architecture}/{fusion_strategy}]"
    # num_items <- it should be passed as a param not injecting like this
    model, loss_weigher = build_model(architecture, fusion_strategy, num_items=num_items)
    optimizer = torch.optim.Adam(list(model.parameters())+list(loss_weigher.parameters()), lr=lr)

    batch_step = BATCH_STEP[architecture]
    train_loader_, val_loader_ = LOADERS[architecture]

    best_val_loss, best_state_dict, best_metrics = float("inf"), None, None
    checks_without_improvement = 0
    history = []

    for epoch in range(epochs):
        model.train()
        for batch in train_loader_:
            losses, _ = batch_step(model, batch)
            loss = loss_weigher(list(losses))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        if (epoch+1) % check_every != 0:
            continue

        model.eval()
        correct, total = 0, 0
        age_err, price_err, sum_insure_err, val_loss_total = 0.0, 0.0, 0.0, 0.0

        with torch.no_grad():
            for batch in val_loader_:
                losses, stats = batch_step(model, batch)
                val_loss_total += loss_weigher(list(losses)).item()
                correct += stats["correct"]
                total += stats["total"]
                age_err += stats["age_err"]
                price_err += stats["price_err"]
                sum_insure_err += stats["sum_insure_err"]

        current_val_loss = val_loss_total / len(val_loader_)
        metrics = {
            "epoch": epoch+1,
            "val_loss": current_val_loss,
            "item_acc": correct / total,
            "age_mae": age_err / total,
            "price_mae": price_err / total,
            "sum_insure_mae": sum_insure_err / total,
            "weights": task_weights(loss_weigher),
        }
        history.append(metrics)
        if verbose:
            w = metrics["weights"]
            print(
                f"{tag} epoch {epoch+1:3d} | val_loss {current_val_loss:.4f} | item_acc {metrics['item_acc']:.2%} "
                f"| age_mae {metrics['age_mae']:.2f} | price_mae {metrics['price_mae']:.2f} | sum_insure_mae {metrics['sum_insure_mae']:.2f}"
            )
            print(f"{tag}   weights: item={w['item']:.2f} age={w['age']:.2f} price={w['price']:.2f} sum_insure={w['sum_insure']:.2f}")

        if current_val_loss < best_val_loss:
            best_val_loss = current_val_loss
            best_state_dict = {k: v.clone() for k, v in model.state_dict().items()}
            best_metrics = metrics
            checks_without_improvement = 0
        else:
            checks_without_improvement += 1
            if checks_without_improvement >= patience:
                if verbose:
                    print(f"{tag} no improvement for {patience} checks - stopping early at epoch {epoch+1}")
                break
    
    model.load_state_dict(best_state_dict)
    return {
        "architecture": architecture,
        "fusion": fusion_strategy,
        "model": model,
        "best_metrics": best_metrics,
        "history": history
    }


In [96]:
result = run_experiment(architecture="decoder", fusion_strategy="sum")
print(result["best_metrics"])

[decoder/sum] epoch   5 | val_loss 3.1981 | item_acc 17.03% | age_mae 1.27 | price_mae 1461.52 | sum_insure_mae 26085.20
[decoder/sum]   weights: item=0.95 age=1.05 price=0.96 sum_insure=0.97
[decoder/sum] epoch  10 | val_loss 3.0827 | item_acc 15.22% | age_mae 1.26 | price_mae 1513.46 | sum_insure_mae 25446.10
[decoder/sum]   weights: item=0.91 age=1.11 price=0.93 sum_insure=0.96
[decoder/sum] epoch  15 | val_loss 3.0591 | item_acc 15.58% | age_mae 1.26 | price_mae 1430.77 | sum_insure_mae 26740.35
[decoder/sum]   weights: item=0.87 age=1.18 price=0.90 sum_insure=0.96
[decoder/sum] epoch  20 | val_loss 3.0526 | item_acc 17.03% | age_mae 1.26 | price_mae 1524.75 | sum_insure_mae 27512.47
[decoder/sum]   weights: item=0.83 age=1.24 price=0.88 sum_insure=0.97
[decoder/sum] epoch  25 | val_loss 3.1404 | item_acc 19.57% | age_mae 1.26 | price_mae 1471.59 | sum_insure_mae 28527.59
[decoder/sum]   weights: item=0.80 age=1.32 price=0.87 sum_insure=0.98
[decoder/sum] epoch  30 | val_loss 3.249

# Experiments

In [101]:
import itertools
import pandas as pd
torch.manual_seed(55)

architectures = [
    "encoder", 
    "decoder"
]
fusion_strategies = ["sum", "concat_all", "concat_emb_summed"]

all_results = {}
for architecture, fusion_strategy in itertools.product(architectures, fusion_strategies):
    print(f"\n{'='*70}\n{architecture}+{fusion_strategy}\n{'='*70}")
    all_results[(architecture, fusion_strategy)] = run_experiment(architecture, fusion_strategy)


encoder+sum
[encoder/sum] epoch   5 | val_loss 2.9862 | item_acc 20.29% | age_mae 1.27 | price_mae 1414.01 | sum_insure_mae 27693.05
[encoder/sum]   weights: item=0.82 age=1.25 price=0.89 sum_insure=0.95
[encoder/sum] epoch  10 | val_loss 2.8423 | item_acc 16.30% | age_mae 1.27 | price_mae 1387.14 | sum_insure_mae 26505.00
[encoder/sum]   weights: item=0.71 age=1.57 price=0.83 sum_insure=0.94
[encoder/sum] epoch  15 | val_loss 2.8044 | item_acc 14.49% | age_mae 1.26 | price_mae 1436.61 | sum_insure_mae 28388.66
[encoder/sum]   weights: item=0.64 age=1.96 price=0.79 sum_insure=0.94
[encoder/sum] epoch  20 | val_loss 2.7071 | item_acc 16.67% | age_mae 1.27 | price_mae 1478.26 | sum_insure_mae 29200.74
[encoder/sum]   weights: item=0.59 age=2.44 price=0.78 sum_insure=0.97
[encoder/sum] epoch  25 | val_loss 2.6237 | item_acc 16.30% | age_mae 1.27 | price_mae 1575.68 | sum_insure_mae 28013.13
[encoder/sum]   weights: item=0.57 age=3.04 price=0.80 sum_insure=1.03
[encoder/sum] epoch  30 | v

In [102]:
rows = []
for (architecture, fusion_strategy), result in all_results.items():
    m = result["best_metrics"]
    rows.append({
        "architecture": architecture,
        "fusion": fusion_strategy,
        "best_epoch": m["epoch"],
        "val_loss": round(m["val_loss"], 4),
        "item_acc": round(m["item_acc"], 4),
        "age_mae": round(m["age_mae"], 2),
        "price_mae": round(m["price_mae"], 2),
        "sum_insure_mae": round(m["sum_insure_mae"], 2),
    })

In [103]:
summary = pd.DataFrame(rows).sort_values(["architecture", "fusion"]).reset_index(drop=True)
summary.to_csv("./runs/experiment_summary.csv", index=False)
summary.sort_values(by=['item_acc'], ascending=[False])

,architecture,fusion,best_epoch,val_loss,item_acc,age_mae,price_mae,sum_insure_mae
1,decoder,concat_emb_summed,30,2.9766,0.1993,1.27,1527.67,25872.87
3,encoder,concat_all,40,2.3003,0.1667,1.27,1498.06,25641.05
0,decoder,concat_all,30,2.9486,0.1594,1.25,1562.45,26407.55
4,encoder,concat_emb_summed,25,2.5773,0.1558,1.27,1532.56,27131.27
5,encoder,sum,30,2.6081,0.1486,1.28,1481.43,27745.58
2,decoder,sum,30,2.9685,0.1413,1.29,1533.61,26975.46


In [104]:
import pickle
with open("./runs/experiment_histories.pkl", "wb") as f:
    pickle.dump({k:v["history"] for k, v in all_results.items()}, f)

# Appendices

In [105]:
import torch
from package.gpt_decoder_block import GPTDecoderBlock

torch.manual_seed(0)

d_model, n_heads, max_len = 16, 4, 8
decoder = GPTDecoderBlock(d_model, n_heads, max_len)
decoder.eval()   # no dropout in this block, but good habit regardless

# one sequence of 8 "events" (a..h) — random vectors standing in for whatever
# FusionEmbedding would have produced. The specific values don't matter — what
# matters is that the SAME decoder weights process them both ways below.
full_sequence = torch.randn(1, 8, d_model)   # (batch=1, seq_len=8, d_model)

with torch.no_grad():
    full_out = decoder(full_sequence)   # ONE forward pass over all 8 positions at once

print(f"{'prefix':<10}{'last-pos idx':<14}{'identical?':<12}sample values (first 3 dims)")
for prefix_len in range(1, 9):
    prefix = full_sequence[:, :prefix_len]        # e.g. prefix_len=3 -> just [a, b, c]
    with torch.no_grad():
        prefix_out = decoder(prefix)               # a SEPARATE forward pass, just this prefix

    from_full   = full_out[:, prefix_len - 1]       # position (prefix_len-1), read from the FULL pass
    from_prefix = prefix_out[:, -1]                 # the LAST position of the truncated pass
    same = torch.allclose(from_full, from_prefix, atol=1e-6)

    label = "abcdefgh"[:prefix_len]
    print(f"{label:<10}{prefix_len-1:<14}{str(same):<12}"
          f"full={from_full[0,:3].tolist()}  prefix={from_prefix[0,:3].tolist()}")


prefix    last-pos idx  identical?  sample values (first 3 dims)
a         0             True        full=[1.2655940055847168, 0.5423235297203064, -1.056722640991211]  prefix=[1.2655938863754272, 0.542323648929596, -1.0567227602005005]
ab        1             True        full=[-1.814671516418457, 0.24059291183948517, 0.2435184270143509]  prefix=[-1.8146713972091675, 0.24059295654296875, 0.24351856112480164]
abc       2             True        full=[-1.7201393842697144, 1.3883235454559326, 0.6254863142967224]  prefix=[-1.720139503479004, 1.3883236646652222, 0.6254865527153015]
abcd      3             True        full=[-2.6280617713928223, 1.356939435005188, 0.7863212823867798]  prefix=[-2.6280617713928223, 1.356939435005188, 0.7863212823867798]
abcde     4             True        full=[0.238181933760643, -0.2198280245065689, 0.7137393951416016]  prefix=[0.238181933760643, -0.2198280245065689, 0.7137393951416016]
abcdef    5             True        full=[-0.1150452047586441, 1.0679932832

In [108]:
full_out.shape, full_sequence.shape

(torch.Size([1, 8, 16]), torch.Size([1, 8, 16]))